# Sprint 4 — Modelo Baseline (v2)
## Proyecto: Productividad Asesores de Negocios

---

### Cambios respecto a v1

- Target con 3 clases: ALTO, BAJO, MEDIO
- Pipeline actualizado con variables de evento (Sprint 3 v2)
- Piso de métricas de referencia actualizado


## 1. Configuración y carga

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    f1_score, cohen_kappa_score, roc_auc_score,
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, make_scorer,
)

warnings.filterwarnings('ignore')

PROCESSED  = Path('../data/processed')
FIG_DIR    = Path('../reports/figures')
MLRUNS_DIR = Path('../mlruns')
FIG_DIR.mkdir(parents=True, exist_ok=True)

data          = np.load(PROCESSED / 'features_processed.npz', allow_pickle=True)
X_train       = data['X_train']
X_test        = data['X_test']
y_train       = data['y_train']
y_test        = data['y_test']
label_encoder = joblib.load(PROCESSED / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)

DB_PATH_MLFLOW = Path('../mlruns/mlflow.db')
DB_PATH_MLFLOW.parent.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f'sqlite:///{DB_PATH_MLFLOW.resolve()}')
mlflow.set_experiment('advisor_risk_classification')

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Clases : {CLASS_NAMES}')
print(f'y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}')


X_train: (1603, 22)
X_test : (401, 22)
Clases : ['ALTO', 'BAJO', 'MEDIO']
y_train: {np.int64(0): np.int64(535), np.int64(1): np.int64(534), np.int64(2): np.int64(534)}


## 2. Funciones de evaluación

In [2]:
def calcular_metricas(y_true, y_pred, y_prob, class_names):
    metricas = {
        'macro_f1' : f1_score(y_true, y_pred, average='macro'),
        'kappa'    : cohen_kappa_score(y_true, y_pred),
        'auc_ovr'  : roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'),
        'accuracy' : accuracy_score(y_true, y_pred),
    }
    f1_por_clase = f1_score(y_true, y_pred, average=None)
    for i, cls in enumerate(class_names):
        metricas[f'f1_{cls}'] = f1_por_clase[i]
    return metricas


def plot_confusion_matrix(y_true, y_pred, class_names, titulo, filepath):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()


print('Funciones de evaluacion definidas.')


Funciones de evaluacion definidas.


## 3. Validación cruzada y entrenamiento

In [3]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    C=1.0,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Ejecutando validacion cruzada (5 folds)...')
cv_results = cross_validate(
    lr_model, X_train, y_train,
    cv=cv,
    scoring={
        'macro_f1' : 'f1_macro',
        'kappa'    : make_scorer(cohen_kappa_score),
        'accuracy' : 'accuracy',
    },
    return_train_score=False,
    n_jobs=-1,
)

print(f'\n=== VALIDACION CRUZADA — LOGISTIC REGRESSION ===')
print(f'  Macro F1 : {cv_results["test_macro_f1"].mean():.4f} +/- {cv_results["test_macro_f1"].std():.4f}')
print(f'  Kappa    : {cv_results["test_kappa"].mean():.4f} +/- {cv_results["test_kappa"].std():.4f}')
print(f'  Accuracy : {cv_results["test_accuracy"].mean():.4f} +/- {cv_results["test_accuracy"].std():.4f}')
print(f'  F1 folds : {cv_results["test_macro_f1"].round(4)}')


Ejecutando validacion cruzada (5 folds)...

=== VALIDACION CRUZADA — LOGISTIC REGRESSION ===
  Macro F1 : 0.6656 +/- 0.0375
  Kappa    : 0.4965 +/- 0.0547
  Accuracy : 0.6644 +/- 0.0365
  F1 folds : [0.7027 0.6949 0.6018 0.6836 0.6452]


In [4]:
with mlflow.start_run(run_name='logistic_regression_baseline_v2') as run:
    mlflow.log_params({
        'model_type'   : 'LogisticRegression',
        'class_weight' : 'balanced',
        'C'            : 1.0,
        'solver'       : 'lbfgs',
        'max_iter'     : 1000,
        'n_classes'    : 3,
        'n_train'      : len(X_train),
        'n_test'       : len(X_test),
        'n_features'   : X_train.shape[1],
        'target'       : 'TARGET_B_3clases',
        'cv_folds'     : 5,
    })
    mlflow.log_metrics({
        'cv_macro_f1_mean': cv_results['test_macro_f1'].mean(),
        'cv_macro_f1_std' : cv_results['test_macro_f1'].std(),
        'cv_kappa_mean'   : cv_results['test_kappa'].mean(),
    })

    lr_model.fit(X_train, y_train)
    y_pred = lr_model.predict(X_test)
    y_prob = lr_model.predict_proba(X_test)

    metricas_test = calcular_metricas(y_test, y_pred, y_prob, CLASS_NAMES)
    mlflow.log_metrics({f'test_{k}': v for k, v in metricas_test.items()})

    cm_path = FIG_DIR / '04_confusion_matrix_lr_baseline_v2.png'
    plot_confusion_matrix(
        y_test, y_pred, CLASS_NAMES,
        'Logistic Regression Baseline v2 — 3 clases', cm_path
    )
    mlflow.log_artifact(str(cm_path))
    mlflow.sklearn.log_model(lr_model, 'logistic_regression_baseline_v2')
    run_id = run.info.run_id

print('=== METRICAS EN TEST — LR BASELINE v2 ===')
print(f'  Macro F1 : {metricas_test["macro_f1"]:.4f}  <- METRICA PRINCIPAL')
print(f'  Kappa    : {metricas_test["kappa"]:.4f}')
print(f'  AUC OvR  : {metricas_test["auc_ovr"]:.4f}')
print(f'  Accuracy : {metricas_test["accuracy"]:.4f}')
print(f'\nF1 por clase:')
for cls in CLASS_NAMES:
    print(f'  {cls:<10}: {metricas_test[f"f1_{cls}"]:.4f}')
print(f'\nMLflow run_id: {run_id}')

print(f'\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))


2026/06/05 12:33:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/05 12:33:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 12:33:41 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


=== METRICAS EN TEST — LR BASELINE v2 ===
  Macro F1 : 0.7107  <- METRICA PRINCIPAL
  Kappa    : 0.5661
  AUC OvR  : 0.8554
  Accuracy : 0.7107

F1 por clase:
  ALTO      : 0.7912
  BAJO      : 0.7209
  MEDIO     : 0.6199

MLflow run_id: d90b3d6f799e4d5681cabf68338405e8

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        ALTO     0.7714    0.8120    0.7912       133
        BAJO     0.7500    0.6940    0.7209       134
       MEDIO     0.6131    0.6269    0.6199       134

    accuracy                         0.7107       401
   macro avg     0.7115    0.7110    0.7107       401
weighted avg     0.7114    0.7107    0.7105       401



## 4. Piso de métricas establecido

In [5]:
print('''
=================================================================
  PISO DE METRICAS — LR BASELINE v2 (3 clases)
  Referencia para comparacion en Sprint 5
=================================================================
''')
print(f'  CV Macro F1 : {cv_results["test_macro_f1"].mean():.4f} +/- {cv_results["test_macro_f1"].std():.4f}')
print(f'  Test Macro F1: {metricas_test["macro_f1"]:.4f}')
print(f'  Test Kappa   : {metricas_test["kappa"]:.4f}')
print(f'  Test AUC OvR : {metricas_test["auc_ovr"]:.4f}')
print()
print('  LightGBM debe superar este baseline en al menos 3pp de macro F1.')
print('  Si no lo supera, documentar por que la complejidad no se justifica.')



  PISO DE METRICAS — LR BASELINE v2 (3 clases)
  Referencia para comparacion en Sprint 5

  CV Macro F1 : 0.6656 +/- 0.0375
  Test Macro F1: 0.7107
  Test Kappa   : 0.5661
  Test AUC OvR : 0.8554

  LightGBM debe superar este baseline en al menos 3pp de macro F1.
  Si no lo supera, documentar por que la complejidad no se justifica.
